In [ ]:
"""
This script implements a machine learning pipeline for classifying handwritten digits from the MNIST dataset using
the Random Greedy Forest (RGF) algorithm. The pipeline includes data loading, image augmentation, model training,
evaluation, and result visualization.

The code performs the following steps:

1. **Setup**: The script installs the `rgf_python` library, which is required for the RGFClassifier.

2. **Load and Augment MNIST Dataset**: The MNIST dataset is fetched from OpenML, and the images are reshaped
    into a 28x28 format. A function `augment_image` is defined to perform basic augmentations on the images,
    including horizontal and vertical flips, as well as a 90-degree clockwise rotation. The original and augmented
    images are collected into new lists.

3. **Train/Test Split**: The augmented dataset is split into training and test sets using an 80/20 ratio, ensuring
    stratification based on the labels.

4. **Train RGFClassifier**: An instance of the RGFClassifier is created with specified hyperparameters, and the model
     is trained on the training dataset.

5. **Learning Curve Visualization**: The learning curve is plotted to visualize the training and validation accuracy
     as a function of the training size.

6. **Model Evaluation**: The model is evaluated on both the training and test sets. Mean Squared Error (MSE) and Mean
     Absolute Error (MAE) are calculated and printed. A confusion matrix is displayed, and a classification report is
     generated to assess model performance.

7. **Feature Importance Visualization**: The script calculates and visualizes the importance of the top 20 features
     (pixels) used by the model.

8. **Model Saving**: The trained model is saved to a file for future use.

9. **Results Packaging**: The model and visualizations are zipped into a single file for download.

10. **Shutdown Runtime**: The script includes a command to shut down the Colab runtime after the file download is
      triggered.

Dependencies:
- numpy
- matplotlib
- opencv-python
- joblib
- scikit-learn
- rgf_python
- google.colab

Usage:
Run the script in a Python environment with the required libraries installed. The output will include model training results, visualizations, and a downloadable zip file containing the trained model and plots.
"""


# STEP 0: Setup
!pip install -q rgf_python

import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import joblib
import zipfile
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, mean_squared_error, mean_absolute_error
)
from rgf.sklearn import RGFClassifier
from google.colab import files

# STEP 1: Load and Augment MNIST
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist["data"], mnist["target"].astype(np.uint8)
X = X.reshape(-1, 28, 28)

def augment_image(img):
    return [cv2.flip(img, 1), cv2.flip(img, 0), cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)]

X_aug, y_aug = [], []
for i in range(len(X)):
    img = X[i]
    label = y[i]
    X_aug.append(img)
    y_aug.append(label)
    for aug in augment_image(img):
        X_aug.append(aug)
        y_aug.append(label)
    if (i + 1) % 10000 == 0:
        print(f"Processed: {i+1} / {len(X)}")

X_aug = np.array(X_aug).reshape(-1, 784)
y_aug = np.array(y_aug)

# STEP 2: Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug, test_size=0.2, stratify=y_aug, random_state=42
)

# STEP 3: Train RGFClassifier
clf = RGFClassifier(max_leaf=500, n_iter=50, min_samples_leaf=10, algorithm="RGF", verbose=True)
clf.fit(X_train, y_train)

# STEP 4: Learning Curve
train_sizes, train_scores, val_scores = learning_curve(
    clf, X_train, y_train, cv=3, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1
)
train_acc = train_scores.mean(axis=1)
val_acc = val_scores.mean(axis=1)

plt.plot(train_sizes, train_acc, label="Train Accuracy")
plt.plot(train_sizes, val_acc, label="Validation Accuracy")
plt.xlabel("Training Size")
plt.ylabel("Accuracy")
plt.title("RGFClassifier Learning Curve")
plt.grid(True)
plt.legend()
plt.savefig("rgf_classifier_learning_curve.png")
plt.close()

# STEP 5: Evaluation
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

print(f"\nTrain MSE: {mean_squared_error(y_train, y_train_pred):.4f}")
print(f"Train MAE: {mean_absolute_error(y_train, y_train_pred):.4f}")
print(f"Test MSE: {mean_squared_error(y_test, y_test_pred):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap="Oranges")
plt.title("RGFClassifier Test Confusion Matrix")
plt.savefig("rgf_classifier_confusion_matrix.png")
plt.close()

# Classification Report
report = classification_report(y_test, y_test_pred, output_dict=False)
print("\nClassification Report (Test):\n")
print(report)

# STEP 6: Feature Importance
importances = clf.feature_importances_
top_idx = np.argsort(importances)[-20:][::-1]
top_vals = importances[top_idx]
feature_names = [f"pixel_{i}" for i in top_idx]

plt.figure(figsize=(10, 6))
plt.barh(feature_names, top_vals)
plt.gca().invert_yaxis()
plt.title("Top 20 RGF Feature Importances")
plt.xlabel("Importance")
plt.grid(True)
plt.tight_layout()
plt.savefig("rgf_classifier_feature_importance.png")
plt.close()

# Save model
joblib.dump(clf, "rgf_classifier_model.joblib")

# STEP 7: Zip Results
zip_filename = "rgf_outputs.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
    for file in [
        "rgf_classifier_model.joblib",
        "rgf_classifier_learning_curve.png",
        "rgf_classifier_confusion_matrix.png",
        "rgf_classifier_feature_importance.png"
    ]:
        zipf.write(file)

# Trigger download
files.download(zip_filename)

# STEP 8: Shut down Colab runtime
import IPython
print("✅ Training complete. Runtime will shut down after file is downloaded.")
IPython.display.display(IPython.display.Javascript('''
  async function shutdown() {
    await google.colab.kernel.invokeFunction('shutdown');
  }
  shutdown();
'''))
